In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split


def find_project_root(start_path: Path) -> Path:
    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (candidate / "src" / "config.py").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root containing src/config.py."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR

PROCESSED_DIR = Path(PROCESSED_DATA_DIR)

VALIDATION_FILE = (
    PROCESSED_DIR / "dataset_validation.csv"
)

SPLIT_FILE = (
    PROCESSED_DIR / "dataset_split.csv"
)

print("Python executable:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Validation file:", VALIDATION_FILE)
print("Validation file exists:", VALIDATION_FILE.exists())

if not VALIDATION_FILE.exists():
    raise FileNotFoundError(
        "dataset_validation.csv was not found. "
        "Run 02_dataset_validation.ipynb first."
    )

Python executable: d:\Projects\music-genre-classification-AI\.venv\Scripts\python.exe
Project root: D:\Projects\music-genre-classification-AI
Validation file: D:\Projects\music-genre-classification-AI\data\processed\dataset_validation.csv
Validation file exists: True


In [2]:
dataset_df = pd.read_csv(VALIDATION_FILE)

required_columns = {
    "file_path",
    "filename",
    "genre",
}

missing_columns = required_columns - set(dataset_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

dataset_df = (
    dataset_df
    .drop_duplicates(subset=["file_path"])
    .reset_index(drop=True)
)

print("Total usable files:", len(dataset_df))
print("Unique file paths:", dataset_df["file_path"].nunique())
print("Number of genres:", dataset_df["genre"].nunique())

genre_counts = (
    dataset_df["genre"]
    .value_counts()
    .sort_index()
    .rename("file_count")
    .to_frame()
)

display(genre_counts)

Total usable files: 999
Unique file paths: 999
Number of genres: 10


,file_count
genre,
blues,100
classical,100
country,100
disco,100
hiphop,100
jazz,99
metal,100
pop,100
reggae,100


In [3]:
train_df, temporary_df = train_test_split(
    dataset_df,
    test_size=0.30,
    random_state=42,
    shuffle=True,
    stratify=dataset_df["genre"],
)

print("Training files:", len(train_df))
print("Temporary files:", len(temporary_df))

Training files: 699
Temporary files: 300


In [4]:
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=temporary_df["genre"],
)

print("Training files:", len(train_df))
print("Validation files:", len(validation_df))
print("Testing files:", len(test_df))

Training files: 699
Validation files: 150
Testing files: 150


In [5]:
train_df = train_df.copy()
validation_df = validation_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
validation_df["split"] = "validation"
test_df["split"] = "test"

split_df = pd.concat(
    [
        train_df,
        validation_df,
        test_df,
    ],
    ignore_index=True,
)

split_df = (
    split_df
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

display(split_df.head())

,file_path,filename,genre,size_bytes,status,sample_rate,channels,frames,duration_seconds,format,subtype,peak_amplitude,rms,warnings,error_type,error_message,split
0,data\raw\genres_original\country\country.00061...,country.00061.wav,country,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,0.721375,0.113724,NaN,NaN,NaN,train
1,data\raw\genres_original\classical\classical.0...,classical.00046.wav,classical,1323564,valid,22050.0,1.0,661760.0,30.011791,WAV,PCM_16,0.636322,0.116659,NaN,NaN,NaN,validation
2,data\raw\genres_original\country\country.00005...,country.00005.wav,country,1333684,valid,22050.0,1.0,666820.0,30.241270,WAV,PCM_16,0.683350,0.207434,NaN,NaN,NaN,train
3,data\raw\genres_original\disco\disco.00049.wav,disco.00049.wav,disco,1323052,valid,22050.0,1.0,661504.0,30.000181,WAV,PCM_16,1.000000,0.226151,NaN,NaN,NaN,train
4,data\raw\genres_original\hiphop\hiphop.00022.wav,hiphop.00022.wav,hiphop,1323632,valid,22050.0,1.0,661794.0,30.013333,WAV,PCM_16,1.000000,0.190738,NaN,NaN,NaN,validation


In [6]:
split_summary = pd.crosstab(
    split_df["genre"],
    split_df["split"],
)

split_summary = split_summary.reindex(
    columns=["train", "validation", "test"],
    fill_value=0,
)

split_summary["total"] = (
    split_summary.sum(axis=1)
)

display(split_summary)

split,train,validation,test,total
genre,,,,
blues,70,15,15,100
classical,70,15,15,100
country,70,15,15,100
disco,70,15,15,100
hiphop,70,15,15,100
jazz,69,15,15,99
metal,70,15,15,100
pop,70,15,15,100
reggae,70,15,15,100


In [8]:
train_paths = set(
    split_df.loc[
        split_df["split"] == "train",
        "file_path",
    ]
)

validation_paths = set(
    split_df.loc[
        split_df["split"] == "validation",
        "file_path",
    ]
)

test_paths = set(
    split_df.loc[
        split_df["split"] == "test",
        "file_path",
    ]
)

assert train_paths.isdisjoint(validation_paths)
assert train_paths.isdisjoint(test_paths)
assert validation_paths.isdisjoint(test_paths)

assert len(split_df) == split_df["file_path"].nunique()

print("No data leakage detected.")
print("Every audio file belongs to exactly one split.")

No data leakage detected.
Every audio file belongs to exactly one split.


In [9]:
missing_paths = []

for file_path in split_df["file_path"]:
    path = Path(file_path)

    if not path.is_absolute():
        path = PROJECT_ROOT / path

    if not path.exists():
        missing_paths.append(str(path))

print("Missing audio files:", len(missing_paths))

if missing_paths:
    display(
        pd.DataFrame(
            {"missing_file_path": missing_paths}
        ).head(20)
    )
else:
    print("All audio files exist.")

Missing audio files: 0
All audio files exist.


In [10]:
columns_to_save = [
    "file_path",
    "filename",
    "genre",
    "sample_rate",
    "channels",
    "frames",
    "duration_seconds",
    "size_bytes",
    "status",
    "warnings",
    "split",
]

available_columns = [
    column
    for column in columns_to_save
    if column in split_df.columns
]

split_df[available_columns].to_csv(
    SPLIT_FILE,
    index=False,
)

print("Dataset split saved to:")
print(SPLIT_FILE)
print("File exists:", SPLIT_FILE.exists())
print("Saved rows:", len(split_df))

Dataset split saved to:
D:\Projects\music-genre-classification-AI\data\processed\dataset_split.csv
File exists: True
Saved rows: 999


In [11]:
saved_split_df = pd.read_csv(SPLIT_FILE)

print("Saved rows:", len(saved_split_df))
print("\nSplit counts:")

display(
    saved_split_df["split"]
    .value_counts()
    .reindex(["train", "validation", "test"])
    .rename("file_count")
    .to_frame()
)

display(saved_split_df.head())

Saved rows: 999

Split counts:


,file_count
split,
train,699
validation,150
test,150


,file_path,filename,genre,sample_rate,channels,frames,duration_seconds,size_bytes,status,warnings,split
0,data\raw\genres_original\country\country.00061...,country.00061.wav,country,22050.0,1.0,661794.0,30.013333,1323632,valid,NaN,train
1,data\raw\genres_original\classical\classical.0...,classical.00046.wav,classical,22050.0,1.0,661760.0,30.011791,1323564,valid,NaN,validation
2,data\raw\genres_original\country\country.00005...,country.00005.wav,country,22050.0,1.0,666820.0,30.241270,1333684,valid,NaN,train
3,data\raw\genres_original\disco\disco.00049.wav,disco.00049.wav,disco,22050.0,1.0,661504.0,30.000181,1323052,valid,NaN,train
4,data\raw\genres_original\hiphop\hiphop.00022.wav,hiphop.00022.wav,hiphop,22050.0,1.0,661794.0,30.013333,1323632,valid,NaN,validation
